In [1]:
from enum import Enum

import numpy as np
import tvm

from tvm.script import tir as T
from tvm.script import tirx as Tx
from tvm.tirx.bench.utils import ProtonContext, bench, export_to_perfetto_trace, CudaProfiler

from tvm.tirx.op_schedule.cuda.copy_async import tma_shared_layout, SwizzleMode
from tvm.tir.layout import TileLayout, TLane, TCol, tid_in_wg


WARM_UP = 10
REPEAT = 30
DEBUG = False


def ceildiv(a, b):
    return (a + b - 1) // b


def prepare_data(M, N, K):
    import torch

    A_f16 = torch.randn((M, K), dtype=torch.float16)
    B_f16 = torch.randn((N, K), dtype=torch.float16)
    C_empty = torch.zeros((M, N), dtype=torch.float16)

    return A_f16, B_f16, C_empty


def cublas_gemm(A_f16, B_f16):
    import torch

    torch_dev = torch.device("cuda")
    A_torch = A_f16.to(torch_dev)
    B_torch = B_f16.to(torch_dev)
    func = lambda: torch.matmul(A_torch, B_torch.T)
    bench(func, warmup=WARM_UP, repeat=REPEAT, proton_name="cublas", debug=DEBUG)
    C_torch = func()
    return C_torch.cpu().numpy()


def tir_gemm(
    A_bf16,
    B_bf16,
    C_bf16,
    kernel,
    profiler_enabled=False,
    profiler_buffer_size=None,
    export_name=None,
    event_type_names=None,
):
    DEV = tvm.cuda(0)
    A_tvm = tvm.runtime.tensor(A_bf16, device=DEV)
    B_tvm = tvm.runtime.tensor(B_bf16, device=DEV)
    C_tvm = tvm.runtime.tensor(C_bf16, device=DEV)
    if profiler_enabled:
        profiler_buffer = tvm.runtime.empty((profiler_buffer_size,), dtype="uint64", device=DEV)
    target = tvm.target.Target("cuda")
    with target:
        mod = tvm.IRModule({"main": kernel})
        ex = tvm.compile(mod, target=target, tir_pipeline="tirx")
        if profiler_enabled:
            func = lambda: ex(A_tvm, B_tvm, C_tvm, profiler_buffer)
        else:
            func = lambda: ex(A_tvm, B_tvm, C_tvm)
        bench(func, warmup=WARM_UP, repeat=REPEAT, proton_name="tir", debug=DEBUG)
    if profiler_enabled:
        export_to_perfetto_trace(
            profiler_buffer.numpy(),
            export_name,
            event_type_names,
        )
    return C_tvm.numpy()


def test(kernel, M, N, K, **kwargs):
    np.random.seed(0)
    A_bf16, B_bf16, C_bf16 = prepare_data(M, N, K)

    with ProtonContext("blackwell hgemm", debug=DEBUG):
        C_tvm = tir_gemm(A_bf16, B_bf16, C_bf16, kernel, **kwargs)
        C_cublas = cublas_gemm(A_bf16, B_bf16)

    np.testing.assert_allclose(C_tvm, C_cublas, rtol=1e-3, atol=1e-2)

In [2]:
a_type = tvm.DataType("float16")
b_type = tvm.DataType("float16")
d_type = tvm.DataType("float16")
acc_type = tvm.DataType("float32")
M, N, K = 8192, 8192, 8192

In [3]:
########################################################################
# 1. synchronous load + tcgen05mma + synchronous store
########################################################################
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_N, BLK_K))

F16_SIZE = 2
SMEM_SIZE = 1024 + (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE


@T.prim_func(tirx=True)
def hgemm_ver1(
    A: T.Buffer((M, K), a_type), B: T.Buffer((N, K), b_type), D: T.Buffer((M, N), d_type)
):
    # fmt: off
    with T.kernel():
        bx, by = T.cta_id([M // BLK_M, N // BLK_N], parent="kernel")
        wg_id = T.warpgroup_id([1], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        mma_bar = pool.alloc((1,), "uint64")
        pool.move_base_to(1024)
        Asmem = pool.alloc((BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((BLK_N, BLK_K), b_type, layout=B_layout)
        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            T.ptx.mbarrier.init(mma_bar.ptr_to([0]), 1)
        # tmem allocation
        if warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], 
                             layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        m_st = T.meta_var(bx * BLK_M)
        n_st = T.meta_var(by * BLK_N)

        phase_mma = T.local_scalar("int32")
        phase_mma = 0
        descI = T.local_scalar("uint32")
        T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)

        for k in range(K // BLK_K):
            k_st = T.meta_var(k * BLK_K)
            # load, whole thread block is involved
            with T.cta():
                Tx.copy(Asmem[:, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K])
                Tx.copy(Bsmem[:, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K])
            T.cuda.cta_sync()
            T.ptx.tcgen05.fence.after_thread_sync()

            # mma, only one thread issues tcgen05mma
            with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                Tx.gemm_async(tmem[:, :BLK_N], Asmem[:, :], Bsmem[:, :], accum=k != 0, dispatch="tcgen05", cta_group=1, descI=descI)
                T.ptx.tcgen05.commit(mma_bar.ptr_to([0]), cta_group=1)
            
            T.ptx.mbarrier.try_wait(mma_bar.ptr_to([0]), phase_mma)
            phase_mma = phase_mma ^ 1
            T.cuda.cta_sync()

        T.ptx.tcgen05.fence.after_thread_sync()
        
        Dreg = T.alloc_local((BLK_N,), "float32")
        Dreg_f16 = T.alloc_local((BLK_N,), "float16")
        # TMA -> RF
        with T.warpgroup():
            Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
            Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
            T.cuda.cta_sync()
        # RF -> RF(f16) -> GMEM
        with T.thread():
            m_st_thr = T.meta_var(m_st + warp_id * 32 + lane_id)
            Tx.cast(Dreg_f16[:], Dreg[:])
            Tx.copy(D[m_st_thr, n_st: n_st + BLK_N], Dreg_f16[:])

        # dealloc TMEM
        if warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [4]:
test(hgemm_ver1, M, N, K)

178.838 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 357.142 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 714.118 hgemm_ver1_kernel



In [5]:
########################################################################
# 2. Add smem stage in writeback
########################################################################
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = 1024 + (BLK_M * BLK_K + BLK_N * BLK_K + BLK_M * BLK_N) * F16_SIZE


@T.prim_func(tirx=True)
def hgemm_ver2(
    A: T.Buffer((M, K), a_type), B: T.Buffer((N, K), b_type), D: T.Buffer((M, N), d_type)
):
    # fmt: off
    with T.kernel():
        bx, by = T.cta_id([M // BLK_M, N // BLK_N], parent="kernel")
        wg_id = T.warpgroup_id([1], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        mma_bar = pool.alloc((1,), "uint64")
        pool.move_base_to(1024)
        Asmem = pool.alloc((BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)
        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            T.ptx.mbarrier.init(mma_bar.ptr_to([0]), 1)
        # tmem allocation
        if warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], 
                             layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        m_st = T.meta_var(bx * BLK_M)
        n_st = T.meta_var(by * BLK_N)

        phase_mma = T.local_scalar("int32")
        phase_mma = 0
        descI = T.local_scalar("uint32")
        T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)

        for k in range(K // BLK_K):
            k_st = T.meta_var(k * BLK_K)
            # load, whole thread block is involved
            with T.cta():
                Tx.copy(Asmem[:, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K])
                Tx.copy(Bsmem[:, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K])
            T.cuda.cta_sync()
            T.ptx.tcgen05.fence.after_thread_sync()

            # mma, only one thread issues tcgen05mma
            with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                Tx.gemm_async(tmem[:, :BLK_N], Asmem[:, :], Bsmem[:, :], accum=k != 0, dispatch="tcgen05", cta_group=1, descI=descI)
                T.ptx.tcgen05.commit(mma_bar.ptr_to([0]), cta_group=1)
            
            T.ptx.mbarrier.try_wait(mma_bar.ptr_to([0]), phase_mma)
            phase_mma = phase_mma ^ 1
            T.cuda.cta_sync()

        T.ptx.tcgen05.fence.after_thread_sync()
        
        Dreg = T.alloc_local((BLK_N,), "float32")
        Dreg_f16 = T.alloc_local((BLK_N,), "float16")
        # TMA -> RF
        with T.warpgroup():
            Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
            Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
            T.cuda.cta_sync()
        # RF -> RF(f16) -> SMEM
        with T.thread():
            Tx.cast(Dreg_f16[:], Dreg[:])
            Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
        # SMEM -> GMEM
        T.cuda.cta_sync()
        with T.cta():
            Tx.copy(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :])

        # dealloc TMEM
        if warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [6]:
test(hgemm_ver2, M, N, K)

188.002 ROOT
├─ 0.535 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.903 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 375.468 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 750.770 hgemm_ver2_kernel



In [7]:
########################################################################
# 3. use tma load/store
########################################################################
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = 1024 + (BLK_M * BLK_K + BLK_N * BLK_K + BLK_M * BLK_N) * F16_SIZE


@T.prim_func(tirx=True)
def hgemm_ver3(
    A: T.Buffer((M, K), a_type), B: T.Buffer((N, K), b_type), D: T.Buffer((M, N), d_type)
):
    # fmt: off
    with T.kernel():
        bx, by = T.cta_id([M // BLK_M, N // BLK_N], parent="kernel")
        wg_id = T.warpgroup_id([1], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        tma_bar = pool.alloc((1,), "uint64")
        mma_bar = pool.alloc((1,), "uint64")
        pool.move_base_to(1024)
        Asmem = pool.alloc((BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)
        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            T.ptx.mbarrier.init(mma_bar.ptr_to([0]), 1)
            T.ptx.mbarrier.init(tma_bar.ptr_to([0]), 1)
        # tmem allocation
        if warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], 
                             layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        m_st = T.meta_var(bx * BLK_M)
        n_st = T.meta_var(by * BLK_N)

        phase_tma = T.local_scalar("int32")
        phase_mma = T.local_scalar("int32")
        phase_tma = 0
        phase_mma = 0
        descI = T.local_scalar("uint32")
        T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)

        @T.inline
        def tma_load(k_st):
            tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma_bar.ptr_to([0])})
            Tx.copy_async(Asmem[:, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
            Tx.copy_async(Bsmem[:, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
            T.ptx.mbarrier.arrive.expect_tx(T.address_of(tma_bar), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE)
        
        @T.inline
        def mma(accum):
            T.ptx.mbarrier.try_wait(tma_bar.ptr_to([0]), phase_tma)
            phase_tma = phase_tma ^ 1
            T.ptx.tcgen05.fence.after_thread_sync()
            Tx.gemm_async(tmem[:, :BLK_N], Asmem[:, :], Bsmem[:, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
            T.ptx.tcgen05.commit(mma_bar.ptr_to([0]), cta_group=1)
            T.ptx.mbarrier.try_wait(mma_bar.ptr_to([0]), phase_mma)
            phase_mma = phase_mma ^ 1

        with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
            for k in range(K // BLK_K):
                # load, whole thread block is involved
                tma_load(k * BLK_K)
                # mma, only one thread issues tcgen05mma
                mma(k != 0)

        T.cuda.cta_sync()
        T.ptx.tcgen05.fence.after_thread_sync()

        Dreg = T.alloc_local((BLK_N,), "float32")
        Dreg_f16 = T.alloc_local((BLK_N,), "float16")
        # TMA -> RF
        with T.warpgroup():
            Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
            Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
            T.cuda.cta_sync()
        # RF -> RF(f16) -> SMEM
        with T.thread():
            Tx.cast(Dreg_f16[:], Dreg[:])
            Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
        # SMEM -> GMEM
        T.cuda.cta_sync()
        with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
            Tx.copy_async(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :], dispatch="tma")
            T.ptx.cp_async.bulk.commit_group()
            T.ptx.cp_async.bulk.wait_group(0)

        # dealloc TMEM
        if warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [8]:
test(hgemm_ver3, M, N, K)

1.751 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 2.969 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 5.771 hgemm_ver3_kernel



In [9]:
########################################################################
# 4. use software pipeline
########################################################################
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

PIPE_DEPTH = 4
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE
)


@T.prim_func(tirx=True)
def hgemm_ver4(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        bx, by = T.cta_id([M // BLK_M, N // BLK_N], parent="kernel")
        wg_id = T.warpgroup_id([1], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32", align=64)
        tma_bar = pool.alloc((PIPE_DEPTH,), "uint64", align=64) # align is required for mbarrier
        mma_bar = pool.alloc((1,), "uint64", align=64)
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        pool.move_base_to(1024) # Dsmem can be reused
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)
        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            T.ptx.mbarrier.init(mma_bar.ptr_to([0]), 1)
            for i in range(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma_bar.ptr_to([i]), 1)
        # tmem allocation
        if warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], 
                             layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        m_st = T.meta_var(bx * BLK_M)
        n_st = T.meta_var(by * BLK_N)

        phase_tma = T.alloc_local((PIPE_DEPTH,), "int32")
        phase_mma = T.local_scalar("int32")
        for i in range(PIPE_DEPTH):
            phase_tma[i] = 0
        phase_mma = 0
        descI = T.local_scalar("uint32")
        T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)

        @T.inline
        def tma_load(stage, k_st):
            tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma_bar.ptr_to([stage])})
            Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
            Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
            T.ptx.mbarrier.arrive.expect_tx(tma_bar.ptr_to([stage]), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE)
        
        @T.inline
        def mma(stage, accum):
            T.ptx.mbarrier.try_wait(tma_bar.ptr_to([stage]), phase_tma[stage])
            phase_tma[stage] = phase_tma[stage] ^ 1
            T.ptx.tcgen05.fence.after_thread_sync()
            Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
            T.ptx.tcgen05.commit(mma_bar.ptr_to([0]), cta_group=1)
            T.ptx.mbarrier.try_wait(mma_bar.ptr_to([0]), phase_mma)
            phase_mma = phase_mma ^ 1

        with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
            for k in range(PRE_NUM):
                tma_load(k, k * BLK_K)

            k_load = T.local_scalar("int32")
            k_load = PRE_NUM
            for k in range(K // BLK_K):
                mma(k % PIPE_DEPTH, k != 0)
                if k_load < K // BLK_K:
                    tma_load(k_load % PIPE_DEPTH, k_load * BLK_K)
                    k_load = k_load + 1

        T.cuda.cta_sync()
        T.ptx.tcgen05.fence.after_thread_sync()

        Dreg = T.alloc_local((BLK_N,), "float32")
        Dreg_f16 = T.alloc_local((BLK_N,), "float16")
        # TMA -> RF
        with T.warpgroup():
            Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
            Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
            T.cuda.cta_sync()
        # RF -> RF(f16) -> SMEM
        with T.thread():
            Tx.cast(Dreg_f16[:], Dreg[:])
            Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
        # SMEM -> GMEM
        T.cuda.cta_sync()
        with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
            Tx.copy_async(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :], dispatch="tma")
            T.ptx.cp_async.bulk.commit_group()
            T.ptx.cp_async.bulk.wait_group(0)

        # dealloc TMEM
        if warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [10]:
test(hgemm_ver4, M, N, K)

1.128 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 1.722 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 3.278 hgemm_ver4_kernel



In [11]:
########################################################################
# 5. persistent kernel
########################################################################
from tvm.tirx.tile_scheduler import ClusterPersistentScheduler2D

BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

PIPE_DEPTH = 4
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE + (BLK_M * BLK_N) * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200


@T.prim_func(tirx=True)
def hgemm_ver5(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([1], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32", align=64)
        tma_bar = pool.alloc((PIPE_DEPTH,), "uint64", align=64) # align is required for mbarrier
        mma_bar = pool.alloc((1,), "uint64", align=64)
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        pool.move_base_to(1024)
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)
        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            T.ptx.mbarrier.init(mma_bar.ptr_to([0]), 1)
            for i in range(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma_bar.ptr_to([i]), 1)
        # tmem allocation
        if warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], 
                             layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // BLK_M, num_n_tiles=N // BLK_N, l2_group_size=8, num_clusters=SM_COUNT))
        tile_scheduler.init(bx)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var(m_idx * BLK_M)
        n_st = T.meta_var(n_idx * BLK_N)

        while tile_scheduler.valid():
            phase_tma = T.alloc_local((PIPE_DEPTH,), "int32")
            phase_mma = T.local_scalar("int32")
            for i in range(PIPE_DEPTH):
                phase_tma[i] = 0
            phase_mma = 0
            descI = T.local_scalar("uint32")
            T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)

            @T.inline
            def tma_load(stage, k_st):
                tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma_bar.ptr_to([stage])})
                Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                T.ptx.mbarrier.arrive.expect_tx(tma_bar.ptr_to([stage]), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE)
            
            @T.inline
            def mma(stage, accum):
                T.ptx.mbarrier.try_wait(tma_bar.ptr_to([stage]), phase_tma[stage])
                phase_tma[stage] = phase_tma[stage] ^ 1
                T.ptx.tcgen05.fence.after_thread_sync()
                Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
                T.ptx.tcgen05.commit(mma_bar.ptr_to([0]), cta_group=1)
                T.ptx.mbarrier.try_wait(mma_bar.ptr_to([0]), phase_mma)
                phase_mma = phase_mma ^ 1

            with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                for k in range(PRE_NUM):
                    tma_load(k, k * BLK_K)

                k_load = T.local_scalar("int32")
                k_load = PRE_NUM
                for k in range(K // BLK_K):
                    mma(k % PIPE_DEPTH, k != 0)
                    if k_load < K // BLK_K:
                        tma_load(k_load % PIPE_DEPTH, k_load * BLK_K)
                        k_load = k_load + 1

            T.cuda.cta_sync()
            T.ptx.tcgen05.fence.after_thread_sync()

            Dreg = T.alloc_local((BLK_N,), "float32")
            Dreg_f16 = T.alloc_local((BLK_N,), "float16")
            # TMA -> RF
            with T.warpgroup():
                Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
                Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
                T.cuda.cta_sync()
            # RF -> RF(f16) -> SMEM
            with T.thread():
                Tx.cast(Dreg_f16[:], Dreg[:])
                Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
            # SMEM -> GMEM
            T.cuda.cta_sync()
            with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                Tx.copy_async(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :], dispatch="tma")
                T.ptx.cp_async.bulk.commit_group()
                T.ptx.cp_async.bulk.wait_group(0)
            
            T.cuda.cta_sync()
            tile_scheduler.next_tile()

        # dealloc TMEM
        if warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [12]:
test(hgemm_ver5, M, N, K)

1.100 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 1.665 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 3.164 hgemm_ver5_kernel



In [13]:
########################################################################
# 6. initial warp specialization
#
# WG1 (warp0: mma, warp3: load), WG0 (writeback)
########################################################################

BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

PIPE_DEPTH = 6
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE + (BLK_M * BLK_N) * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200
WG_NUMBER = 2  # WG1 (warp0: mma, warp3: load), WG0 (writeback)


@T.prim_func(tirx=True)
def hgemm_ver6(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([WG_NUMBER], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        
        tma2mma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2tma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2ld = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        ld2mma = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)

        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            for i in range(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma2mma.ptr_to([i]), 1)
                T.ptx.mbarrier.init(mma2tma.ptr_to([i]), 1)
            T.ptx.mbarrier.init(mma2ld.ptr_to([0]), 1)
            T.ptx.mbarrier.init(ld2mma.ptr_to([0]), 128)

        # tmem allocation
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // BLK_M, num_n_tiles=N // BLK_N, l2_group_size=8, num_clusters=SM_COUNT))
        tile_scheduler.init(bx)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var(m_idx * BLK_M)
        n_st = T.meta_var(n_idx * BLK_N)

        if wg_id == 1:
            if warp_id == 3:
                # load warp
                phase_mma2tma = T.local_scalar("int32")
                phase_mma2tma = 1 # start from 1 to avoid waiting for the first tile

                @T.inline
                def tma_load_stage(stage, k_st):
                    T.ptx.mbarrier.try_wait(mma2tma.ptr_to([stage]), phase_mma2tma) # wait for the mma to finish
                    tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma2mma.ptr_to([stage])})
                    Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                    T.ptx.mbarrier.arrive.expect_tx(tma2mma.ptr_to([stage]), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE) # signal the start of tma

                @T.inline
                def tma_load():
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            tma_load_stage(ks, (ko * PIPE_DEPTH + ks) * BLK_K)
                        phase_mma2tma = phase_mma2tma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        tma_load_stage(ks, (PIPE_CYCLE * PIPE_DEPTH + ks) * BLK_K)
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        T.ptx.mbarrier.try_wait(mma2tma.ptr_to([ks]), phase_mma2tma)
                        T.ptx.mbarrier.arrive(tma2mma.ptr_to([ks]))
                    phase_mma2tma = phase_mma2tma ^ 1
                
                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        tma_load()
                        tile_scheduler.next_tile()

            elif warp_id == 0:
                # mma warp
                phase_tma2mma = T.local_scalar("int32")
                phase_tma2mma = 0
                phase_ld2mma = T.local_scalar("int32")
                phase_ld2mma = 1 # start from 1 to avoid waiting for the first tile

                descI = T.local_scalar("uint32")
                T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)
                T.ptx.tcgen05.fence.after_thread_sync()

                @T.inline
                def mma_stage(stage, accum):
                    T.ptx.mbarrier.try_wait(tma2mma.ptr_to([stage]), phase_tma2mma) # wait for the tma to finish                    
                    Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
                    T.ptx.tcgen05.commit(mma2tma.ptr_to([stage]), cta_group=1) # signal the start of mma

                @T.inline
                def mma():
                    T.ptx.mbarrier.try_wait(ld2mma.ptr_to([0]), phase_ld2mma) # wait for the ld to finish
                    phase_ld2mma = phase_ld2mma ^ 1
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            mma_stage(ks, not (PIPE_CYCLE > 0 and ko == 0 and ks == 0))
                        phase_tma2mma = phase_tma2mma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        mma_stage(ks, not (PIPE_CYCLE == 0 and ks == 0))
                    T.ptx.tcgen05.commit(mma2ld.ptr_to([0]), cta_group=1) # signal the finish of mma
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        T.ptx.mbarrier.try_wait(tma2mma.ptr_to([ks]), phase_tma2mma)
                        T.ptx.mbarrier.arrive(mma2tma.ptr_to([ks]))
                    phase_tma2mma = phase_tma2mma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        mma()
                        tile_scheduler.next_tile()

        elif wg_id == 0:
            phase_mma2ld = T.local_scalar("int32")
            phase_mma2ld = 0

            @T.inline
            def writeback():
                T.ptx.mbarrier.try_wait(mma2ld.ptr_to([0]), phase_mma2ld) # wait for the mma to finish
                phase_mma2ld = phase_mma2ld ^ 1

                T.ptx.tcgen05.fence.after_thread_sync()
                Dreg = T.alloc_local((BLK_N,), "float32")
                Dreg_f16 = T.alloc_local((BLK_N,), "float16")

                # TMA -> RF
                with T.warpgroup():
                    Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
                    Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
                    T.cuda.warpgroup_sync(10)

                T.ptx.mbarrier.arrive(ld2mma.ptr_to([0])) # signal the finish of writeback from TMEM

                # RF -> RF(f16) -> SMEM
                with T.thread():
                    Tx.cast(Dreg_f16[:], Dreg[:])
                    Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
                    T.ptx.fence.proxy_async("shared::cta")
                    T.cuda.warpgroup_sync(10)

                # SMEM -> GMEM
                with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                    Tx.copy_async(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :], dispatch="tma")
                    T.ptx.cp_async.bulk.commit_group()
                    T.ptx.cp_async.bulk.wait_group(0)
                
                T.cuda.warpgroup_sync(10)

            # writeback warpgroup
            while tile_scheduler.valid():
                writeback()
                tile_scheduler.next_tile()

        T.cuda.cta_sync()
        # dealloc TMEM
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [14]:
test(hgemm_ver6, M, N, K)

0.589 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.901 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 0.645 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 1.123 hgemm_ver6_kernel



In [15]:
########################################################################
# 7. visualize the pipeline using profiler, no change to schedule
#
# WG1 (warp0: mma, warp3: load), WG0 (LD_TMEM)
########################################################################

BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

PIPE_DEPTH = 6
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, BLK_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE + (BLK_M * BLK_N) * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200
WG_NUMBER = 2  # WG1 (warp0: mma, warp3: load), WG0 (LD_TMEM)
# profiler related
PROFILER_GROUPS = 3
PROFILER_BUFFER_SIZE = int(8e6)
PROFILER_WRITE_STRIDE = SM_COUNT * PROFILER_GROUPS
PROFILER_ON = True


class ProfileEventType(Enum):
    IssueTMA = 0
    IssueMMA = 1
    LD_TMEM = 2
    WAIT_TMA2MMA = 3
    WAIT_MMA2TMA = 4
    WAIT_MMA2LD = 5
    WAIT_LD2MMA = 6
    WRITEBACK = 7


event_type_names = [
    "issue-tma",
    "issue-mma",
    "ld-tmem",
    "wait-tma2mma",
    "wait-mma2tma",
    "wait-mma2ld",
    "wait-ld2mma",
    "writeback",
]


@T.prim_func(tirx=True)
def hgemm_ver7(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
    profiler_buffer: T.Buffer((PROFILER_BUFFER_SIZE,), "uint64"),
):
    # fmt: off
    with T.kernel():
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([WG_NUMBER], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        
        tma2mma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2tma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2ld = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        ld2mma = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, BLK_N), d_type, layout=D_layout)
        profiler = T.meta_var(CudaProfiler(profiler_buffer, write_stride=PROFILER_WRITE_STRIDE, num_groups=PROFILER_GROUPS, profiler_enabled=PROFILER_ON))

        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            for i in range(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma2mma.ptr_to([i]), 1)
                T.ptx.mbarrier.init(mma2tma.ptr_to([i]), 1)
            T.ptx.mbarrier.init(mma2ld.ptr_to([0]), 1)
            T.ptx.mbarrier.init(ld2mma.ptr_to([0]), 128)

        # tmem allocation
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr[0], layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // BLK_M, num_n_tiles=N // BLK_N, l2_group_size=8, num_clusters=SM_COUNT))
        tile_scheduler.init(bx)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var(m_idx * BLK_M)
        n_st = T.meta_var(n_idx * BLK_N)

        if wg_id == 1:
            if warp_id == 3:
                profiler.init(0)
                # load warp
                phase_mma2tma = T.local_scalar("int32")
                phase_mma2tma = 1 # start from 1 to avoid waiting for the first tile

                @T.inline
                def tma_load_stage(stage, k_st):
                    profiler.start(ProfileEventType.WAIT_MMA2TMA, lane_id == 0)
                    T.ptx.mbarrier.try_wait(mma2tma.ptr_to([stage]), phase_mma2tma) # wait for the mma to finish
                    profiler.end(ProfileEventType.WAIT_MMA2TMA, lane_id == 0)
                    profiler.start(ProfileEventType.IssueTMA, lane_id == 0)
                    tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma2mma.ptr_to([stage])})
                    Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                    T.ptx.mbarrier.arrive.expect_tx(tma2mma.ptr_to([stage]), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE) # signal the start of tma
                    profiler.end(ProfileEventType.IssueTMA, lane_id == 0)

                @T.inline
                def tma_load():
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            tma_load_stage(ks, (ko * PIPE_DEPTH + ks) * BLK_K)
                        phase_mma2tma = phase_mma2tma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        tma_load_stage(ks, (PIPE_CYCLE * PIPE_DEPTH + ks) * BLK_K)
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        profiler.start(ProfileEventType.WAIT_MMA2TMA, lane_id == 0)
                        T.ptx.mbarrier.try_wait(mma2tma.ptr_to([ks]), phase_mma2tma)
                        profiler.end(ProfileEventType.WAIT_MMA2TMA, lane_id == 0)
                        profiler.start(ProfileEventType.IssueTMA, lane_id == 0)
                        T.ptx.mbarrier.arrive(tma2mma.ptr_to([ks]))
                        profiler.end(ProfileEventType.IssueTMA, lane_id == 0)
                    phase_mma2tma = phase_mma2tma ^ 1
                
                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        tma_load()
                        tile_scheduler.next_tile()

            elif warp_id == 0:
                profiler.init(1)
                # mma warp
                phase_tma2mma = T.local_scalar("int32")
                phase_tma2mma = 0
                phase_ld2mma = T.local_scalar("int32")
                phase_ld2mma = 1 # start from 1 to avoid waiting for the first tile

                descI = T.local_scalar("uint32")
                T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)
                T.ptx.tcgen05.fence.after_thread_sync()

                @T.inline
                def mma_stage(stage, accum):
                    profiler.start(ProfileEventType.WAIT_TMA2MMA, lane_id == 0)
                    T.ptx.mbarrier.try_wait(tma2mma.ptr_to([stage]), phase_tma2mma) # wait for the tma to finish                    
                    profiler.end(ProfileEventType.WAIT_TMA2MMA, lane_id == 0)
                    profiler.start(ProfileEventType.IssueMMA, lane_id == 0)
                    Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
                    T.ptx.tcgen05.commit(mma2tma.ptr_to([stage]), cta_group=1) # signal the start of mma
                    profiler.end(ProfileEventType.IssueMMA, lane_id == 0)

                @T.inline
                def mma():
                    profiler.start(ProfileEventType.WAIT_LD2MMA, lane_id == 0)
                    T.ptx.mbarrier.try_wait(ld2mma.ptr_to([0]), phase_ld2mma) # wait for the ld to finish
                    profiler.end(ProfileEventType.WAIT_LD2MMA, lane_id == 0)
                    phase_ld2mma = phase_ld2mma ^ 1
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            mma_stage(ks, not (PIPE_CYCLE > 0 and ko == 0 and ks == 0))
                        phase_tma2mma = phase_tma2mma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        mma_stage(ks, not (PIPE_CYCLE == 0 and ks == 0))
                    T.ptx.tcgen05.commit(mma2ld.ptr_to([0]), cta_group=1) # signal the finish of mma
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        profiler.start(ProfileEventType.WAIT_TMA2MMA, lane_id == 0)
                        T.ptx.mbarrier.try_wait(tma2mma.ptr_to([ks]), phase_tma2mma)
                        profiler.end(ProfileEventType.WAIT_TMA2MMA, lane_id == 0)
                        T.ptx.mbarrier.arrive(mma2tma.ptr_to([ks]))
                    phase_tma2mma = phase_tma2mma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        mma()
                        tile_scheduler.next_tile()

        elif wg_id == 0:
            profiler.init(2)
            phase_mma2ld = T.local_scalar("int32")
            phase_mma2ld = 0

            @T.inline
            def writeback():
                profiler.start(ProfileEventType.WAIT_MMA2LD, warp_id == 0 and lane_id == 0)
                T.ptx.mbarrier.try_wait(mma2ld.ptr_to([0]), phase_mma2ld) # wait for the mma to finish
                profiler.end(ProfileEventType.WAIT_MMA2LD, warp_id == 0 and lane_id == 0)
                phase_mma2ld = phase_mma2ld ^ 1

                T.ptx.tcgen05.fence.after_thread_sync()
                Dreg = T.alloc_local((BLK_N,), "float32")
                Dreg_f16 = T.alloc_local((BLK_N,), "float16")

                # TMA -> RF
                profiler.start(ProfileEventType.LD_TMEM, warp_id == 0 and lane_id == 0)
                with T.warpgroup():
                    Dreg_wg = Dreg.view(128, BLK_N, layout=TileLayout(([128, BLK_N], [1@tid_in_wg, 1])))
                    Tx.copy(Dreg_wg[:, :], tmem[:, :BLK_N])
                T.cuda.warpgroup_sync(10)
                profiler.end(ProfileEventType.LD_TMEM, warp_id == 0 and lane_id == 0)

                T.ptx.mbarrier.arrive(ld2mma.ptr_to([0])) # signal the finish of LD_TMEM from TMEM

                profiler.start(ProfileEventType.WRITEBACK, warp_id == 0 and lane_id == 0)
                # RF -> RF(f16) -> SMEM
                with T.thread():
                    Tx.cast(Dreg_f16[:], Dreg[:])
                    Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[:])
                T.ptx.fence.proxy_async("shared::cta")
                T.cuda.warpgroup_sync(10)
                # SMEM -> GMEM
                with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                    Tx.copy_async(D[m_st: m_st + BLK_M, n_st: n_st + BLK_N], Dsmem[:, :], dispatch="tma")
                    T.ptx.cp_async.bulk.commit_group()
                    T.ptx.cp_async.bulk.wait_group(0)
                profiler.end(ProfileEventType.WRITEBACK, warp_id == 0 and lane_id == 0)
                T.cuda.warpgroup_sync(10)

            # LD_TMEM warpgroup
            while tile_scheduler.valid():
                writeback()
                tile_scheduler.next_tile()

        T.cuda.cta_sync()
        # dealloc TMEM
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [16]:
test(
    hgemm_ver7,
    M,
    N,
    K,
    profiler_enabled=True,
    export_name="hgemm_ver7.perfetto-trace",
    event_type_names=event_type_names,
    profiler_buffer_size=PROFILER_BUFFER_SIZE,
)

0.806 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.901 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 1.079 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 1.992 hgemm_ver7_kernel



In [17]:
########################################################################
# 8. Make write back stage use less registers and shared memory
########################################################################
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 128, 128, 16

PIPE_DEPTH = 6
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)


EPI_N = 64
TMEM_LD_N = 8


A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, EPI_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE + (BLK_M * EPI_N) * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200
WG_NUMBER = 2  # WG1 (warp0: mma, warp3: load), WG0 (LD_TMEM)


@T.prim_func(tirx=True)
def hgemm_ver8(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([WG_NUMBER], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        
        tma2mma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2tma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2ld = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        ld2mma = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, EPI_N), d_type, layout=D_layout)

        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            for i in range(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma2mma.ptr_to([i]), 1)
                T.ptx.mbarrier.init(mma2tma.ptr_to([i]), 1)
            T.ptx.mbarrier.init(mma2ld.ptr_to([0]), 1)
            T.ptx.mbarrier.init(ld2mma.ptr_to([0]), 128)

        # tmem allocation
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=1)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cta_sync()

        tmem_addr_local = T.local_scalar("uint32")
        tmem_addr_local = tmem_addr[0]
        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr_local, layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // BLK_M, num_n_tiles=N // BLK_N, l2_group_size=8, num_clusters=SM_COUNT))
        tile_scheduler.init(bx)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var(m_idx * BLK_M)
        n_st = T.meta_var(n_idx * BLK_N)

        if wg_id == 1:
            if warp_id == 3:
                # load warp
                phase_mma2tma = T.local_scalar("int32")
                phase_mma2tma = 1 # start from 1 to avoid waiting for the first tile

                @T.inline
                def tma_load_stage(stage, k_st):
                    T.ptx.mbarrier.try_wait(mma2tma.ptr_to([stage]), phase_mma2tma) # wait for the mma to finish
                    tma_config = T.meta_var({"dispatch": "tma", "cta_group": 1, "mbar": tma2mma.ptr_to([stage])})
                    Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                    T.ptx.mbarrier.arrive.expect_tx(tma2mma.ptr_to([stage]), (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE) # signal the start of tma

                @T.inline
                def tma_load():
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            tma_load_stage(ks, (ko * PIPE_DEPTH + ks) * BLK_K)
                        phase_mma2tma = phase_mma2tma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        tma_load_stage(ks, (PIPE_CYCLE * PIPE_DEPTH + ks) * BLK_K)
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        T.ptx.mbarrier.try_wait(mma2tma.ptr_to([ks]), phase_mma2tma)
                        T.ptx.mbarrier.arrive(tma2mma.ptr_to([ks]))
                    phase_mma2tma = phase_mma2tma ^ 1
                
                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        tma_load()
                        tile_scheduler.next_tile()

            elif warp_id == 0:
                # mma warp
                phase_tma2mma = T.local_scalar("int32")
                phase_tma2mma = 0
                phase_ld2mma = T.local_scalar("int32")
                phase_ld2mma = 1 # start from 1 to avoid waiting for the first tile

                descI = T.local_scalar("uint32")
                T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, 1)
                T.ptx.tcgen05.fence.after_thread_sync()

                @T.inline
                def mma_stage(stage, accum):
                    T.ptx.mbarrier.try_wait(tma2mma.ptr_to([stage]), phase_tma2mma) # wait for the tma to finish                    
                    Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=1, descI=descI)
                    T.ptx.tcgen05.commit(mma2tma.ptr_to([stage]), cta_group=1) # signal the start of mma

                @T.inline
                def mma():
                    T.ptx.mbarrier.try_wait(ld2mma.ptr_to([0]), phase_ld2mma) # wait for the ld to finish
                    phase_ld2mma = phase_ld2mma ^ 1
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.serial(PIPE_DEPTH):
                            mma_stage(ks, not (PIPE_CYCLE > 0 and ko == 0 and ks == 0))
                        phase_tma2mma = phase_tma2mma ^ 1
                    for ks in T.serial(PIPE_REMAIN_NUM):
                        mma_stage(ks, not (PIPE_CYCLE == 0 and ks == 0))
                    T.ptx.tcgen05.commit(mma2ld.ptr_to([0]), cta_group=1) # signal the finish of mma
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        T.ptx.mbarrier.try_wait(tma2mma.ptr_to([ks]), phase_tma2mma)
                        T.ptx.tcgen05.commit(mma2tma.ptr_to([ks]), cta_group=1)
                    phase_tma2mma = phase_tma2mma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        mma()
                        tile_scheduler.next_tile()

        elif wg_id == 0:
            phase_mma2ld = T.local_scalar("int32")
            phase_mma2ld = 0

            @T.inline
            def writeback():
                T.ptx.mbarrier.try_wait(mma2ld.ptr_to([0]), phase_mma2ld) # (both CTAs) wait for the issues of all mmas of current tile to finish
                phase_mma2ld = phase_mma2ld ^ 1
                T.ptx.tcgen05.fence.after_thread_sync()

                Dreg_f16 = T.alloc_local((MMA_N,), "float16")
                for no in T.unroll(MMA_N // TMEM_LD_N):
                    Dreg = T.alloc_local((TMEM_LD_N,), "float32")
                    with T.warpgroup():
                        Dreg_wg = Dreg.view(128, TMEM_LD_N, layout=TileLayout(([128, TMEM_LD_N], [1@tid_in_wg, 1])))
                        Tx.copy(Dreg_wg[:, :], tmem[:, no * TMEM_LD_N : no * TMEM_LD_N + TMEM_LD_N])
                    with T.thread():
                        Tx.cast(Dreg_f16[no * TMEM_LD_N : no * TMEM_LD_N + TMEM_LD_N], Dreg[:])
                            
                # signal the finish of LD_TMEM from TMEM
                T.ptx.mbarrier.arrive(ld2mma.ptr_to([0]), cta_id=0, pred=True) # (both CTAs) signal (CTA-0) the finish of LD_TMEM from TMEM, such that CTA-0 can proceed to issue next mma
                    
                for no in T.unroll(MMA_N // EPI_N):
                    with T.thread():
                        Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[no * EPI_N : (no + 1) * EPI_N])
                        T.ptx.fence.proxy_async("shared::cta")
                    T.cuda.warpgroup_sync(10)

                    # smem -> gmem
                    with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                        n_st_epi = T.meta_var(n_idx * MMA_N + no * EPI_N)
                        Tx.copy_async(D[m_st: m_st + BLK_M, n_st_epi: n_st_epi + EPI_N], Dsmem[:, :], dispatch="tma")
                        T.ptx.cp_async.bulk.commit_group()
                        T.ptx.cp_async.bulk.wait_group(0)
                    T.cuda.warpgroup_sync(10)

            # LD_TMEM warpgroup
            while tile_scheduler.valid():
                writeback()
                tile_scheduler.next_tile()

        T.cuda.cta_sync()
        # dealloc TMEM
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=1)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=1)

In [18]:
test(hgemm_ver8, M, N, K)

0.587 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 0.640 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 1.115 hgemm_ver8_kernel



In [19]:
########################################################################
# 9. Cluster (2-CTA cooperation)
########################################################################
from tvm.ir import PointerType, PrimType

CTA_GROUP = 2
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 256, 256, 16

PIPE_DEPTH = 6
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

EPI_N = 64
TMEM_LD_N = 8

A_layout = tma_shared_layout(a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_M, BLK_K))
B_layout = tma_shared_layout(b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K))
D_layout = tma_shared_layout(d_type, SwizzleMode.SWIZZLE_128B_ATOM, (BLK_M, EPI_N))

F16_SIZE = 2
SMEM_SIZE = (
    1024 + PIPE_DEPTH * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE + (BLK_M * EPI_N) * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200
WG_NUMBER = 2  # WG1 (warp0: mma, warp3: load), WG0 (LD_TMEM)


@T.prim_func(tirx=True)
def hgemm_ver9(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        cbx, cby = T.cta_id([CTA_GROUP, 1], parent="cluster")
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([WG_NUMBER], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        
        tma2mma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2tma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2ld = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        ld2mma = pool.alloc((1,), "uint64", align=8) # align is required for mbarrier
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((BLK_M, EPI_N), d_type, layout=D_layout)

        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            for i in T.unroll(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma2mma.ptr_to([i]), 1)
                T.ptx.mbarrier.init(mma2tma.ptr_to([i]), 1)
            T.ptx.mbarrier.init(mma2ld.ptr_to([0]), 1)
            T.ptx.mbarrier.init(ld2mma.ptr_to([0]), 128 * CTA_GROUP)

        # CTA-0 is responsible for receiving signals of the finish of TMA of both 2 CTAs
        ptr: T.Var(name="ptr", dtype=PointerType(PrimType("uint64"))) = T.reinterpret("handle", T.ptx.map_shared_rank(tma2mma.ptr_to([0]), 0))
        tma2mma_cta0 = T.decl_buffer([CTA_GROUP], "uint64", data=ptr, scope="shared")

        # tmem allocation
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=CTA_GROUP)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cluster_sync()
        
        tmem_addr_local = T.local_scalar("uint32")
        tmem_addr_local = tmem_addr[0]

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=tmem_addr_local, layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // MMA_M, num_n_tiles=N // MMA_N, l2_group_size=8, num_clusters=SM_COUNT // CTA_GROUP))
        tile_scheduler.init(bx // CTA_GROUP)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var((m_idx * CTA_GROUP + cbx) * BLK_M)
        n_st = T.meta_var((n_idx * CTA_GROUP + cbx) * BLK_N)

        if wg_id == 1:
            if warp_id == 3:
                # load warp
                phase_mma2tma = T.local_scalar("int32")
                phase_mma2tma = 1 # start from 1 to avoid waiting for the first tile

                @T.inline
                def tma_load_stage(stage, k_st):
                    T.ptx.mbarrier.try_wait(mma2tma.ptr_to([stage]), phase_mma2tma) # both CTAs wait for the mma (issued by CTA-0)to finish]
                    tma_config = T.meta_var({"dispatch": "tma", "cta_group": CTA_GROUP, "mbar": tma2mma_cta0.ptr_to([stage])})
                    Tx.copy_async(Asmem[stage, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                    if cbx == 0:
                        T.ptx.mbarrier.arrive.expect_tx(tma2mma_cta0.ptr_to([stage]), CTA_GROUP * (BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE) # signal CTA-0 the issue of tma

                @T.inline
                def tma_load():
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.unroll(PIPE_DEPTH):
                            tma_load_stage(ks, (ko * PIPE_DEPTH + ks) * BLK_K)
                        phase_mma2tma = phase_mma2tma ^ 1
                    for ks in T.unroll(PIPE_REMAIN_NUM):
                        tma_load_stage(ks, (PIPE_CYCLE * PIPE_DEPTH + ks) * BLK_K)
                    for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                        T.ptx.mbarrier.try_wait(mma2tma.ptr_to([ks]), phase_mma2tma)
                        if cbx == 0:
                            T.ptx.mbarrier.arrive(tma2mma.ptr_to([ks]))
                    phase_mma2tma = phase_mma2tma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        tma_load()
                        tile_scheduler.next_tile()

            elif warp_id == 0:
                if cbx == 0: # only CTA-0 is responsible for issuing mma
                    # mma warp
                    phase_tma2mma = T.local_scalar("int32")
                    phase_tma2mma = 0
                    phase_ld2mma = T.local_scalar("int32")
                    phase_ld2mma = 1 # start from 1 to avoid waiting for the first tile

                    descI = T.local_scalar("uint32")
                    T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, CTA_GROUP)
                    T.ptx.tcgen05.fence.after_thread_sync()

                    @T.inline
                    def mma_stage(stage, accum):
                        T.ptx.mbarrier.try_wait(tma2mma.ptr_to([stage]), phase_tma2mma) # wait for the tma to finish
                        Tx.gemm_async(tmem[:, :BLK_N], Asmem[stage, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=CTA_GROUP, descI=descI)
                        T.ptx.tcgen05.commit(mma2tma.ptr_to([stage]), cta_group=CTA_GROUP, cta_mask=3) # signal (both CTAs) the issue of mma

                    @T.inline
                    def mma():
                        T.ptx.mbarrier.try_wait(ld2mma.ptr_to([0]), phase_ld2mma) # wait for the ld (of both CTAs) to finish
                        phase_ld2mma = phase_ld2mma ^ 1
                        for ko in T.serial(PIPE_CYCLE):
                            for ks in T.unroll(PIPE_DEPTH):
                                mma_stage(ks, not (PIPE_CYCLE > 0 and ko == 0 and ks == 0))
                            phase_tma2mma = phase_tma2mma ^ 1
                        for ks in T.unroll(PIPE_REMAIN_NUM):
                            mma_stage(ks, not (PIPE_CYCLE == 0 and ks == 0))
                        T.ptx.tcgen05.commit(mma2ld.ptr_to([0]), cta_group=CTA_GROUP, cta_mask=3) # signal (both CTAs) the issue of all mmas of current tile
                        for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                            T.ptx.mbarrier.try_wait(tma2mma.ptr_to([ks]), phase_tma2mma)
                            T.ptx.tcgen05.commit(mma2tma.ptr_to([ks]), cta_group=CTA_GROUP, cta_mask=3)
                        phase_tma2mma = phase_tma2mma ^ 1

                    with T.thread(parent="warp")[T.ptx.elect_sync()]:
                        while tile_scheduler.valid():
                            mma()
                            tile_scheduler.next_tile()

        elif wg_id == 0:
            phase_mma2ld = T.local_scalar("int32")
            phase_mma2ld = 0

            @T.inline
            def writeback():
                T.ptx.mbarrier.try_wait(mma2ld.ptr_to([0]), phase_mma2ld) # (both CTAs) wait for the issues of all mmas of current tile to finish
                phase_mma2ld = phase_mma2ld ^ 1
                T.ptx.tcgen05.fence.after_thread_sync()

                Dreg_f16 = T.alloc_local((MMA_N,), "float16")
                for no in T.unroll(MMA_N // TMEM_LD_N):
                    Dreg = T.alloc_local((TMEM_LD_N,), "float32")
                    with T.warpgroup():
                        Dreg_wg = Dreg.view(128, TMEM_LD_N, layout=TileLayout(([128, TMEM_LD_N], [1@tid_in_wg, 1])))
                        Tx.copy(Dreg_wg[:, :], tmem[:, no * TMEM_LD_N : no * TMEM_LD_N + TMEM_LD_N])
                    with T.thread():
                        Tx.cast(Dreg_f16[no * TMEM_LD_N : no * TMEM_LD_N + TMEM_LD_N], Dreg[:])
                            
                # signal the finish of LD_TMEM from TMEM
                T.ptx.mbarrier.arrive(ld2mma.ptr_to([0]), cta_id=0, pred=True) # (both CTAs) signal (CTA-0) the finish of LD_TMEM from TMEM, such that CTA-0 can proceed to issue next mma
                    
                for no in T.unroll(MMA_N // EPI_N):
                    with T.thread():
                        Tx.copy(Dsmem[warp_id * 32 + lane_id, :], Dreg_f16[no * EPI_N : (no + 1) * EPI_N])
                        T.ptx.fence.proxy_async("shared::cta")
                    T.cuda.warpgroup_sync(10)

                    # smem -> gmem
                    with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                        n_st_epi = T.meta_var(n_idx * MMA_N + no * EPI_N)
                        Tx.copy_async(D[m_st: m_st + BLK_M, n_st_epi: n_st_epi + EPI_N], Dsmem[:, :], dispatch="tma")
                        T.ptx.cp_async.bulk.commit_group()
                        T.ptx.cp_async.bulk.wait_group(0)
                    T.cuda.warpgroup_sync(10)

            # LD_TMEM warpgroup
            while tile_scheduler.valid():
                writeback()
                tile_scheduler.next_tile()

        T.cuda.cluster_sync()
        # dealloc TMEM
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=CTA_GROUP)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=CTA_GROUP)

In [20]:
test(hgemm_ver9, M, N, K)

0.535 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 0.537 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 0.907 hgemm_ver9_kernel



In [21]:
########################################################################
# 10. 2 consumer warp specialization scheme
# WG2 (warp0: mma, warp1: mma, warp3: load), WG0 (LD_TMEM + writeback), WG1 (LD_TMEM + writeback)
########################################################################
from tvm.ir import PointerType, PrimType

CTA_GROUP = 2
NUM_CONSUMER = 2
BLK_M, BLK_N, BLK_K = 128, 128, 64
MMA_M, MMA_N, MMA_K = 256, 256, 16

PIPE_DEPTH = 4
PIPE_CYCLE = (K // BLK_K) // PIPE_DEPTH
PIPE_REMAIN_NUM = (K // BLK_K) % PIPE_DEPTH
PRE_NUM = min(PIPE_DEPTH, K // BLK_K)

EPI_N = 64
TMEM_LD_N = 8

A_layout = tma_shared_layout(
    a_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, NUM_CONSUMER, BLK_M, BLK_K)
)
B_layout = tma_shared_layout(
    b_type, SwizzleMode.SWIZZLE_128B_ATOM, (PIPE_DEPTH, BLK_N, BLK_K)
)
D_layout = tma_shared_layout(
    d_type, SwizzleMode.SWIZZLE_128B_ATOM, (NUM_CONSUMER, BLK_M, EPI_N)
)

F16_SIZE = 2
SMEM_SIZE = (
    1024
    + PIPE_DEPTH * NUM_CONSUMER * BLK_M * BLK_K * F16_SIZE
    + PIPE_DEPTH * BLK_N * BLK_K * F16_SIZE
    + NUM_CONSUMER * BLK_M * EPI_N * F16_SIZE
)

SM_COUNT = 148  # number of Stream Multiprocessors for B200
WG_NUMBER = 3  # WG2 (warp0: mma, warp1: mma, warp3: load), WG0 (LD_TMEM + writeback), WG1 (LD_TMEM + writeback)


@T.prim_func(tirx=True)
def hgemm_ver10(
    A: T.Buffer((M, K), a_type),
    B: T.Buffer((N, K), b_type),
    D: T.Buffer((M, N), d_type),
):
    # fmt: off
    with T.kernel():
        cbx, cby = T.cta_id([CTA_GROUP, 1], parent="cluster")
        bx = T.cta_id([SM_COUNT], parent="kernel")
        wg_id = T.warpgroup_id([WG_NUMBER], parent="cta")
        warp_id = T.warp_id([4], parent="warpgroup")
        lane_id = T.thread_id([32], parent="warp")

        # smem allocation
        buf = T.alloc_buffer([SMEM_SIZE], "uint8", scope="shared.dyn")
        pool = T.meta_var(Tx.PoolAllocator(buf.data))
        tmem_addr = pool.alloc((1,), "uint32")
        
        tma2mma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2tma = pool.alloc((PIPE_DEPTH,), "uint64", align=8) # align is required for mbarrier
        mma2ld = pool.alloc((NUM_CONSUMER,), "uint64", align=8) # align is required for mbarrier
        ld2mma = pool.alloc((NUM_CONSUMER,), "uint64", align=8) # align is required for mbarrier
        pool.move_base_to(1024)
        Asmem = pool.alloc((PIPE_DEPTH, NUM_CONSUMER, BLK_M, BLK_K), a_type, layout=A_layout)
        Bsmem = pool.alloc((PIPE_DEPTH, BLK_N, BLK_K), b_type, layout=B_layout)
        Dsmem = pool.alloc((NUM_CONSUMER, BLK_M, EPI_N), d_type, layout=D_layout)

        # mbarrier initialization
        if warp_id == 0 and lane_id == 0:
            for i in T.unroll(PIPE_DEPTH):
                T.ptx.mbarrier.init(tma2mma.ptr_to([i]), 1) # singaled by warp3 of CTA-0
                T.ptx.mbarrier.init(mma2tma.ptr_to([i]), NUM_CONSUMER) # signaled by warp0 & 1 of CTA-0
            for i in T.unroll(NUM_CONSUMER):
                T.ptx.mbarrier.init(mma2ld.ptr_to([i]), 1) # signaled by warp0(1) of CTA-0
                T.ptx.mbarrier.init(ld2mma.ptr_to([i]), 128 * CTA_GROUP) # signaled by warpgroup1(2) of both CTAs

        # CTA-0 is responsible for receiving signals of the finish of TMA of both 2 CTAs
        ptr: T.Var(name="ptr", dtype=PointerType(PrimType("uint64"))) = T.reinterpret("handle", T.ptx.map_shared_rank(tma2mma.ptr_to([0]), 0))
        tma2mma_cta0 = T.decl_buffer([PIPE_DEPTH], "uint64", data=ptr, scope="shared")

        # tmem allocation
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.alloc(T.address_of(tmem_addr), n_cols=512, cta_group=CTA_GROUP)

        T.ptx.fence.proxy_async("shared::cta")
        T.ptx.fence.mbarrier_init()
        T.cuda.cluster_sync()
        
        tmem_addr_local = T.local_scalar("uint32")
        tmem_addr_local = tmem_addr[0]

        tmem = T.decl_buffer((128, 512), "float32", scope="tmem", allocated_addr=0, layout=TileLayout(([128, 512], [1@TLane, 1@TCol])))

        tile_scheduler = T.meta_var(ClusterPersistentScheduler2D("tile_scheduler", num_m_tiles=M // MMA_M // NUM_CONSUMER, num_n_tiles=N // MMA_N, l2_group_size=8, num_clusters=SM_COUNT // CTA_GROUP))
        tile_scheduler.init(bx // CTA_GROUP)
        m_idx = T.meta_var(tile_scheduler.m_idx)
        n_idx = T.meta_var(tile_scheduler.n_idx)
        m_st = T.meta_var((m_idx * NUM_CONSUMER * CTA_GROUP + cbx) * BLK_M)
        n_st = T.meta_var((n_idx * CTA_GROUP + cbx) * BLK_N)

        if wg_id == 2:
            if warp_id == 3:
                # load warp
                phase_mma2tma = T.local_scalar("int32")
                phase_mma2tma = 1 # start from 1 to avoid waiting for the first tile

                @T.inline
                def tma_load_stage(stage, k_st):
                    T.ptx.mbarrier.try_wait(mma2tma.ptr_to([stage]), phase_mma2tma) # both CTAs wait for the mma (issued by warp0/1 of CTA-0) to finish]
                    tma_config = T.meta_var({"dispatch": "tma", "cta_group": CTA_GROUP, "mbar": tma2mma_cta0.ptr_to([stage])})
                    Tx.copy_async(Asmem[stage, 0, :, :], A[m_st : m_st + BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Asmem[stage, 1, :, :], A[m_st + CTA_GROUP * BLK_M : m_st + (CTA_GROUP + 1) * BLK_M, k_st : k_st + BLK_K], **tma_config)
                    Tx.copy_async(Bsmem[stage, :, :], B[n_st : n_st + BLK_N, k_st : k_st + BLK_K], **tma_config)
                    if cbx == 0:
                        T.ptx.mbarrier.arrive.expect_tx(tma2mma_cta0.ptr_to([stage]), CTA_GROUP * (NUM_CONSUMER * BLK_M * BLK_K + BLK_N * BLK_K) * F16_SIZE) # signal CTA-0 the issue of tma

                @T.inline
                def tma_load():
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.unroll(PIPE_DEPTH):
                            tma_load_stage(ks, (ko * PIPE_DEPTH + ks) * BLK_K)
                        phase_mma2tma = phase_mma2tma ^ 1
                    if PIPE_REMAIN_NUM > 0:
                        for ks in T.unroll(PIPE_REMAIN_NUM):
                            tma_load_stage(ks, (PIPE_CYCLE * PIPE_DEPTH + ks) * BLK_K)
                        for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                            T.ptx.mbarrier.try_wait(mma2tma.ptr_to([ks]), phase_mma2tma)
                            if cbx == 0:
                                T.ptx.mbarrier.arrive(tma2mma.ptr_to([ks]))
                        phase_mma2tma = phase_mma2tma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        tma_load()
                        tile_scheduler.next_tile()

            elif warp_id < 2 and cbx == 0:
                # mma warp
                phase_tma2mma = T.local_scalar("int32")
                phase_tma2mma = 0
                phase_ld2mma = T.local_scalar("int32")
                phase_ld2mma = 1 # start from 1 to avoid waiting for the first tile

                descI = T.local_scalar("uint32")
                T.ptx.tcgen05.encode_instr_descriptor(T.address_of(descI), acc_type, a_type, b_type, MMA_M, MMA_N, MMA_K, False, False, CTA_GROUP)
                T.ptx.tcgen05.fence.after_thread_sync()

                @T.inline
                def mma_stage(stage, accum):
                    T.ptx.mbarrier.try_wait(tma2mma.ptr_to([stage]), phase_tma2mma) # wait for the tma to finish
                    Tx.gemm_async(tmem[:, warp_id * MMA_N: warp_id * MMA_N + BLK_N], Asmem[stage, warp_id, :, :], Bsmem[stage, :, :], accum=accum, dispatch="tcgen05", cta_group=CTA_GROUP, descI=descI)
                    T.ptx.tcgen05.commit(mma2tma.ptr_to([stage]), cta_group=CTA_GROUP, cta_mask=3) # signal (both CTAs) the issue of mma

                @T.inline
                def mma():
                    T.ptx.mbarrier.try_wait(ld2mma.ptr_to([warp_id]), phase_ld2mma) # wait for the ld of corresponding consumer (of both CTAs) to finish
                    phase_ld2mma = phase_ld2mma ^ 1
                    for ko in T.serial(PIPE_CYCLE):
                        for ks in T.unroll(PIPE_DEPTH):
                            mma_stage(ks, not (PIPE_CYCLE > 0 and ko == 0 and ks == 0))
                        phase_tma2mma = phase_tma2mma ^ 1
                    if PIPE_REMAIN_NUM > 0:
                        for ks in T.unroll(PIPE_REMAIN_NUM):
                            mma_stage(ks, not (PIPE_CYCLE == 0 and ks == 0))
                    T.ptx.tcgen05.commit(mma2ld.ptr_to([warp_id]), cta_group=CTA_GROUP, cta_mask=3) # signal the corresponding consumer (of both CTAs) the issue of all mmas of current tile
                    if PIPE_REMAIN_NUM > 0:
                        for ks in T.unroll(PIPE_REMAIN_NUM, PIPE_DEPTH):
                            T.ptx.mbarrier.try_wait(tma2mma.ptr_to([ks]), phase_tma2mma)
                            T.ptx.tcgen05.commit(mma2tma.ptr_to([ks]), cta_group=CTA_GROUP, cta_mask=3)
                        phase_tma2mma = phase_tma2mma ^ 1

                with T.thread(parent="warp")[T.ptx.elect_sync()]:
                    while tile_scheduler.valid():
                        mma()
                        tile_scheduler.next_tile()

        elif wg_id < 2:
            phase_mma2ld = T.local_scalar("int32")
            phase_mma2ld = 0

            @T.inline
            def writeback():
                T.ptx.mbarrier.try_wait(mma2ld.ptr_to([wg_id]), phase_mma2ld) # wait for the issues of all mmas issued by CTA-0 (warp0(1)) to finish
                phase_mma2ld = phase_mma2ld ^ 1
                T.ptx.tcgen05.fence.after_thread_sync()

                Dreg_f16 = T.alloc_local((MMA_N,), "float16")
                for no in T.unroll(MMA_N // TMEM_LD_N):
                    Dreg = T.alloc_local((TMEM_LD_N,), "float32")
                    with T.warpgroup():
                        Dreg_wg = Dreg.view(128, TMEM_LD_N, layout=TileLayout(([128, TMEM_LD_N], [1@tid_in_wg, 1])))
                        n_tmem_ld_st = T.meta_var(wg_id * MMA_N + no * TMEM_LD_N)
                        Tx.copy(Dreg_wg[:, :], tmem[:, n_tmem_ld_st : n_tmem_ld_st + TMEM_LD_N])
                    with T.thread():
                        Tx.cast(Dreg_f16[no * TMEM_LD_N : no * TMEM_LD_N + TMEM_LD_N], Dreg[:])

                # signal the finish of LD_TMEM from TMEM
                T.ptx.mbarrier.arrive(ld2mma.ptr_to([wg_id]), cta_id=0, pred=True) # signal CTA-0 (warp0(1)) the finish of LD_TMEM from TMEM, such that CTA-0 (warp0(1)) can proceed to issue next mma

                for no in T.unroll(MMA_N // EPI_N):
                    with T.thread():
                        Tx.copy(Dsmem[wg_id, warp_id * 32 + lane_id, :], Dreg_f16[no * EPI_N : (no + 1) * EPI_N])
                        T.ptx.fence.proxy_async("shared::cta")
                    T.cuda.warpgroup_sync(wg_id + 10)

                    # smem -> gmem
                    with T.thread(parent="warpgroup")[warp_id == 0 and lane_id == 0]:
                        m_st_epi = T.meta_var((m_idx * NUM_CONSUMER * CTA_GROUP + wg_id * CTA_GROUP + cbx) * BLK_M)
                        n_st_epi = T.meta_var(n_idx * MMA_N + no * EPI_N)
                        Tx.copy_async(D[m_st_epi: m_st_epi + BLK_M, n_st_epi: n_st_epi + EPI_N], Dsmem[wg_id, :, :], dispatch="tma")
                        T.ptx.cp_async.bulk.commit_group()
                        T.ptx.cp_async.bulk.wait_group(0)
                    T.cuda.warpgroup_sync(wg_id + 10)

            # LD_TMEM warpgroup
            while tile_scheduler.valid():
                writeback()
                tile_scheduler.next_tile()

        T.cuda.cluster_sync()
        # dealloc TMEM
        if wg_id == 0 and warp_id == 0:
            T.ptx.tcgen05.relinquish_alloc_permit(cta_group=CTA_GROUP)
            T.ptx.tcgen05.dealloc(tmem_addr[0], n_cols=512, cta_group=CTA_GROUP)

In [ ]:
test(hgemm_ver10, M, N, K)

0.534 ROOT
├─ 0.534 cublas
│  ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
│  └─ 0.902 nvjet_hsh_256x256_64x4_2x1_2cta_v_bz_TNT
└─ 0.534 tir
   ├─ 0.166 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
   └─ 0.901 hgemm_ver10_kernel



: 